# Cooling Check: Weather-Normalized Building Anomaly Detection

## Framing

### Who it's for
**Facility operators** — people responsible for keeping buildings running efficiently and spotting when cooling systems aren't performing as expected.

### The Question
**Is this building's cooling running worse than the weather explains?**

More formally: given outdoor temperature and humidity, how much cooling energy *should* this building use? When actual usage exceeds that prediction, something's wrong (fouled heat exchanger, valve stuck, compressor inefficiency, etc.).

### What this is NOT
- Not production code or a real-time monitoring system
- Not a perfect physics model (it's simplified, linear)
- Not a replacement for domain expertise or maintenance schedules
- Not a control system or automated setpoint optimizer

---

## Why Weather-Normalize?

**The Problem**: Comparing raw cooling energy month-to-month is misleading.
- If it's hotter, you use more cooling. That's not a problem; that's physics.
- If it's more humid, cooling towers work harder. Again, expected.
- A "20% higher energy use" month might just be a hotter month.

**The Solution**: Build a model that explains cooling load as a function of weather. Then compare actual vs predicted.
- If actual > predicted by 10%, that's an anomaly worth investigating.
- If actual < predicted, maybe the system is over-designed or controls are tight.

**Why this matters for operators**:
- Focuses attention on real problems, not weather noise.
- Enables early detection of degradation before full failure.
- Helps justify maintenance budgets ("we found this before it broke").

---

## Approach

1. **Physics baseline**: Fit a change-point model to relate cooling load to outdoor temperature (ASHRAE Guideline 14).
2. **ML on the gap**: Model the residual (actual − baseline) to capture hour-of-day, day-of-week, humidity effects.
3. **Uncertainty quantification**: Provide prediction bands so operators know the confidence.
4. **Time-series validation**: Backtest on held-out months to ensure we're not overfitting.
5. **Cold-start capability**: Show how to onboard a new building with just 2–4 weeks of data.
6. **Drift detection**: Monitor for slow degradation or equipment changes.
7. **Actionable output**: Flag hours outside the band, convert excess to $ and CO₂.


---

## Section 2: Data

**Why one site?** ASHRAE GEPIII has 16 sites. Picking one keeps the scope tight for a portfolio piece, lets us understand local weather patterns well, and shows we can think about sample bias. (Full production would pool across sites, with site-level random effects.)

**Which site?** Site 0: 24 buildings with chilled water, Education + Office + Lodging mix, ~200K sq ft (large enough to show real patterns, not noise).

**Why these 7 buildings?** Largest by square footage (200K–130K sq ft), diverse use types, good data coverage.

### Raw Data Inspection

Before cleaning, let's look at what we have:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Load data
ASHRAE_PATH = '../ashrae-energy-prediction'
metadata = pd.read_csv(f'{ASHRAE_PATH}/building_metadata.csv')
train_raw = pd.read_csv(f'{ASHRAE_PATH}/train.csv')
weather_raw = pd.read_csv(f'{ASHRAE_PATH}/weather_train.csv')

# Filter to meter 1 (chilled water)
meter_1 = train_raw[train_raw['meter'] == 1].copy()

# Select 7 buildings from Site 0 (largest by sq ft)
site_0_meta = metadata[(metadata['site_id'] == 0) & (metadata['building_id'].isin(meter_1['building_id']))]
SELECTED_BUILDINGS = site_0_meta.nlargest(7, 'square_feet')['building_id'].values
SITE_ID = 0

print(f"Site: {SITE_ID}")
print(f"Selected buildings: {SELECTED_BUILDINGS}")
print(f"\nBuilding details:")
print(site_0_meta.loc[site_0_meta['building_id'].isin(SELECTED_BUILDINGS), 
                      ['building_id', 'primary_use', 'square_feet', 'year_built']])

---

## Section 3: Physics Features

### Data Cleaning Decisions: Summary

| Decision | Reason | Consequence |
|----------|--------|-------------|
| Keep only Site 0 | Manageable scope, understand local weather | Lose generalization; production would pool sites |
| Select 7 buildings (largest by sq ft) | Good signal (large buildings), diverse mix | Selection bias toward modern, big buildings |
| Drop leading zeros | Equipment startup or missing data, not real cooling | Lose Jan–early Mar data, bias toward summer |
| Keep only air_temp & dew_point | Other weather features >50% missing | Lose cloud cover, wind effects (acceptable trade) |
| Merge on inner join (weather ∩ energy) | Both must have data | Lose a few hours with missing weather |
| **No imputation** | Cleaning decision #1: be honest about gaps | Smaller dataset, but no synthetic data |

**Answer out loud:**
1. **Why one site and not all?** Manageable scope for a portfolio piece. Shows I understand sample bias. Production would pool sites with hierarchical effects.
2. **What did you drop, and could that bias results?** Leading zeros (Jan–Feb data → summer bias), buildings <200K sq ft (large-building bias), features with >50% missing (can't use cloud/wind). Summer bias is real—we'll validate on months with full coverage in the backtest (Section 7).

In [ ]:
# Combine all buildings' data with weather
all_data = []
for b_id in SELECTED_BUILDINGS:
    df = cleaned_data[b_id][['timestamp', 'meter_reading']].copy()
    df['building_id'] = b_id
    all_data.append(df)

energy_df = pd.concat(all_data, ignore_index=True).sort_values(['building_id', 'timestamp'])

# Merge with weather
data = energy_df.merge(weather_use, on='timestamp', how='inner')

# Add time features for later
data['hour'] = data['timestamp'].dt.hour
data['day_of_week'] = data['timestamp'].dt.dayofweek
data['month'] = data['timestamp'].dt.month
data['is_weekend'] = data['day_of_week'] >= 5

# Get building metadata
data = data.merge(metadata[['building_id', 'primary_use', 'square_feet']],
                  on='building_id', how='left')

print(f"Final dataset shape: {data.shape}")
print(f"Date range: {data['timestamp'].min()} to {data['timestamp'].max()}")
print(f"Buildings: {data['building_id'].nunique()}")
print(f"Hours per building: {data.groupby('building_id').size()}")
print(f"\nColumns: {data.columns.tolist()}")
print(f"\nFirst rows:")
print(data.head())

### Final Dataset: Merge and Align

In [ ]:
# Weather for Site 0
weather = weather_raw[weather_raw['site_id'] == SITE_ID].copy()
weather['timestamp'] = pd.to_datetime(weather['timestamp'])

# Keep columns needed for physics model
# air_temperature: outdoor dry bulb (input to change-point model)
# dew_temperature: for humidity calculations (wet-bulb, enthalpy)
# Others have too many missing values (cloud_coverage 78% missing)
weather_use = weather[['timestamp', 'air_temperature', 'dew_temperature']].copy()

# Drop rows with missing temperature (we need this for the model)
initial_weather = len(weather_use)
weather_use = weather_use.dropna(subset=['air_temperature', 'dew_temperature'])
dropped_weather = initial_weather - len(weather_use)

print(f"Weather: {len(weather_use)} hours (dropped {dropped_weather} with missing temp)")
print(f"Date range: {weather_use['timestamp'].min()} to {weather_use['timestamp'].max()}")

# Sanity check: temperature range
print(f"\nTemperature range: {weather_use['air_temperature'].min():.1f}°C to {weather_use['air_temperature'].max():.1f}°C")
print(f"Dew point range: {weather_use['dew_temperature'].min():.1f}°C to {weather_use['dew_temperature'].max():.1f}°C")

### Weather Data

Pull weather for Site 0, keeping only the columns we'll use:

In [ ]:
def clean_building_data(building_id, meter_1_df):
    """
    Clean chilled water data for one building.
    Returns: cleaned DataFrame, metadata about the cleaning
    """
    df = meter_1_df[meter_1_df['building_id'] == building_id].copy().sort_values('timestamp')
    
    initial_count = len(df)
    
    # Drop leading zeros (equipment startup or missing data)
    df = df[df['meter_reading'] > 0].reset_index(drop=True)
    leading_zero_count = initial_count - len(df)
    
    # Parse timestamp
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Drop duplicates (same timestamp and building = error in raw data)
    df = df.drop_duplicates(subset=['building_id', 'timestamp'], keep='first')
    dup_count = initial_count - len(df) - leading_zero_count
    
    # Check for gaps > 1 hour (we expect hourly data)
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['hour_gap'] = df['timestamp'].diff().dt.total_seconds() / 3600
    gap_hours = df[df['hour_gap'] > 1.5]
    gap_count = len(gap_hours)
    
    if gap_count > 0:
        print(f"  Building {building_id}: {gap_count} gaps > 1.5 hours")
    
    return df, {
        'initial': initial_count,
        'leading_zeros': leading_zero_count,
        'duplicates': dup_count,
        'gaps': gap_count,
        'final': len(df)
    }

# Clean all selected buildings
cleaned_data = {}
cleaning_log = []

for b_id in SELECTED_BUILDINGS:
    df, log = clean_building_data(b_id, meter_1)
    cleaned_data[b_id] = df
    log['building_id'] = b_id
    cleaning_log.append(log)
    
cleaning_summary = pd.DataFrame(cleaning_log)
print("Cleaning summary:")
print(cleaning_summary[['building_id', 'initial', 'leading_zeros', 'duplicates', 'gaps', 'final']])

### Data Cleaning Decisions

**Decision 1: Handle leading zeros**
- Building 95 has 393 consecutive zeros (Feb 29 – Mar 14)
- **Why**: This looks like equipment startup or missing data, not real zero cooling
- **Action**: Drop all records before first non-zero reading for each building
- **Could this bias results?** Yes—we're biasing toward summer (peak cooling). Mitigation: we'll backtest on months with full data (Section 7).

In [ ]:
# Pick one building and look at raw rows
b_id = SELECTED_BUILDINGS[0]  # Building 95
raw = meter_1[meter_1['building_id'] == b_id].sort_values('timestamp').reset_index(drop=True)

print(f"Building {b_id}: {len(raw)} hours of data")
print(f"Date range: {raw['timestamp'].min()} to {raw['timestamp'].max()}")
print(f"\nFirst 10 hours:")
print(raw.head(10))

print(f"\n--- Check 1: Zero readings ---")
zero_ct = (raw['meter_reading'] == 0).sum()
print(f"Zero readings: {zero_ct} / {len(raw)} ({100*zero_ct/len(raw):.1f}%)")
print("First rows with zeros:")
zero_rows = raw[raw['meter_reading'] == 0].head(20)
print(zero_rows[['timestamp', 'meter_reading']])

print(f"\n--- Check 2: Time gaps ---")
raw['ts'] = pd.to_datetime(raw['timestamp'])
raw['gap'] = raw['ts'].diff().dt.total_seconds() / 3600  # hours
gaps = raw[raw['gap'] > 1.5]
print(f"Records with gap > 1.5 hours: {len(gaps)}")
if len(gaps) > 0:
    print("First 10 gaps:")
    print(gaps[['timestamp', 'gap']].head(10))

print(f"\n--- Check 3: Value distribution ---")
print(raw['meter_reading'].describe())